# Per-Vertical TRIBE Analysis

Stratified breakdown of the headline finding by vertical. **Run after `analyze.ipynb`.**

**Outputs (saved to `data/analysis_outputs/`):**
1. `per_vertical_auc.png` and `.csv` — baseline vs baseline+TRIBE AUC per vertical
2. `per_vertical_diff_maps.png` — viral-dud activation difference, one per vertical

**Important caveat:** n ≈ 10 per vertical (5 viral + 5 dud minus extraction failures). The numbers in this notebook are **directional, not statistically significant**. Frame them as 'patterns by vertical' in the writeup, not 'validation results'.

## 1. Setup and load (re-runs the same loading code as analyze.ipynb)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path('..')
CSV_PATH = PROJECT_ROOT / 'data' / 'videos.csv'
FEATURES_DIR = PROJECT_ROOT / 'data' / 'features'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'analysis_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load metadata + features
meta = pd.read_csv(CSV_PATH)
available_npz = set(f.stem for f in FEATURES_DIR.glob('*.npz'))
meta['video_id'] = meta['video_id'].astype(str)
meta = meta[meta['video_id'].isin(available_npz)].copy().reset_index(drop=True)

fmri_means = []
for vid in meta['video_id']:
    npz = np.load(FEATURES_DIR / f'{vid}.npz', allow_pickle=True)
    fmri_means.append(npz['fmri_preds'].mean(axis=0))
fmri_means = np.stack(fmri_means)

# Baseline features
meta['log_followers'] = np.log1p(pd.to_numeric(meta['creator_followers_at_post'], errors='coerce').fillna(0))
meta['duration_num'] = pd.to_numeric(meta['duration'], errors='coerce').fillna(0)

try:
    from sentence_transformers import SentenceTransformer
    miniLM = SentenceTransformer('all-MiniLM-L6-v2')
    captions = meta['caption'].fillna('').astype(str).tolist()
    caption_emb = miniLM.encode(captions, show_progress_bar=False)
except Exception as e:
    print(f'Caption embeddings failed ({e}); using zero fallback')
    caption_emb = np.zeros((len(meta), 384))

X_baseline = np.hstack([meta[['log_followers', 'duration_num']].values, caption_emb])
X_baseline_plus_tribe = np.hstack([X_baseline, fmri_means])

print(f'Loaded {len(meta)} videos with features')
print(f'Verticals: {sorted(meta["vertical"].unique())}')

## 2. Per-vertical AUC

For each vertical, fit baseline and baseline+TRIBE on just that vertical's viral/dud rows.

Uses 3-fold CV (not 5) because n is small per vertical and we need enough samples per fold for a viral and a dud to coexist in test.

In [ ]:
def cv_auc(X, y, n_splits=3):
    """Stratified CV AUC. Returns (mean, std, n_folds_actually_run)."""
    if len(np.unique(y)) < 2:
        return np.nan, np.nan, 0
    # Make sure n_splits doesn't exceed minimum class count
    min_class = min((y == 0).sum(), (y == 1).sum())
    n_splits = min(n_splits, min_class)
    if n_splits < 2:
        return np.nan, np.nan, 0
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    aucs = []
    for train_idx, test_idx in skf.split(X, y):
        if len(np.unique(y[test_idx])) < 2:
            continue
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[train_idx])
        Xte = scaler.transform(X[test_idx])
        clf = LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE)
        clf.fit(Xtr, y[train_idx])
        proba = clf.predict_proba(Xte)[:, 1]
        aucs.append(roc_auc_score(y[test_idx], proba))
    if not aucs:
        return np.nan, np.nan, 0
    return float(np.mean(aucs)), float(np.std(aucs)), len(aucs)

verticals = sorted(meta['vertical'].unique())
results = []

for v in verticals:
    mask = (meta['vertical'] == v) & (meta['label'].isin(['viral', 'dud']))
    if mask.sum() < 4:
        print(f'{v}: only {mask.sum()} viral/dud samples, skipping')
        continue
    y_v = (meta.loc[mask, 'label'] == 'viral').astype(int).values
    Xb_v = X_baseline[mask.values]
    Xt_v = X_baseline_plus_tribe[mask.values]
    n_viral = int(y_v.sum())
    n_dud = int((1 - y_v).sum())
    auc_b, _, folds_b = cv_auc(Xb_v, y_v)
    auc_t, _, folds_t = cv_auc(Xt_v, y_v)
    delta = (auc_t - auc_b) if not (np.isnan(auc_b) or np.isnan(auc_t)) else np.nan
    results.append({
        'vertical': v,
        'n_viral': n_viral,
        'n_dud': n_dud,
        'auc_baseline': auc_b,
        'auc_tribe': auc_t,
        'delta': delta,
        'folds': folds_b,
    })

per_vertical = pd.DataFrame(results).sort_values('delta', ascending=False).reset_index(drop=True)
print('\nPer-vertical AUC results:')
print(per_vertical.to_string())
per_vertical.to_csv(OUTPUT_DIR / 'per_vertical_auc.csv', index=False)
print(f'\nSaved {OUTPUT_DIR / "per_vertical_auc.csv"}')

In [ ]:
# Plot deltas as a horizontal bar chart
valid = per_vertical.dropna(subset=['delta']).copy()
if len(valid) == 0:
    print('No valid per-vertical results to plot.')
else:
    valid = valid.sort_values('delta')
    fig, ax = plt.subplots(figsize=(9, 0.6 * len(valid) + 2))
    colors = ['steelblue' if d >= 0 else 'firebrick' for d in valid['delta']]
    bars = ax.barh(valid['vertical'], valid['delta'], color=colors, alpha=0.8)
    for bar, row in zip(bars, valid.itertuples()):
        # Annotate with both AUCs and n
        x = bar.get_width()
        x_text = x + (0.005 if x >= 0 else -0.005)
        align = 'left' if x >= 0 else 'right'
        ax.text(x_text, bar.get_y() + bar.get_height() / 2,
                f'  {row.auc_baseline:.2f} → {row.auc_tribe:.2f}  (n={row.n_viral}v/{row.n_dud}d)',
                va='center', ha=align, fontsize=9)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('AUC delta (baseline + TRIBE) − baseline')
    ax.set_title('TRIBE lift over baseline, by vertical\n(directional only — n is small per vertical)')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'per_vertical_auc.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {OUTPUT_DIR / "per_vertical_auc.png"}')

## 3. Per-vertical diff maps

For each vertical: average viral fmri activation minus average dud fmri activation, plotted on the brain.

These are the most visually striking output and probably the best single figure for a pitch.

In [ ]:
from nilearn import plotting, datasets

fsaverage = datasets.fetch_surf_fsaverage('fsaverage5')
n_per_hemi = 10242

# Compute per-vertical diff maps
vertical_diffs = {}
for v in verticals:
    viral_mask = (meta['vertical'] == v) & (meta['label'] == 'viral')
    dud_mask = (meta['vertical'] == v) & (meta['label'] == 'dud')
    if viral_mask.sum() == 0 or dud_mask.sum() == 0:
        print(f'{v}: missing viral or dud, skipping diff map')
        continue
    viral_mean = fmri_means[viral_mask.values].mean(axis=0)
    dud_mean = fmri_means[dud_mask.values].mean(axis=0)
    vertical_diffs[v] = {
        'diff': viral_mean - dud_mean,
        'n_viral': int(viral_mask.sum()),
        'n_dud': int(dud_mask.sum()),
    }

print(f'Computed diff maps for {len(vertical_diffs)} verticals')

# Use a shared color scale across all verticals so the figure is comparable
all_diffs = np.stack([d['diff'] for d in vertical_diffs.values()])
shared_max = np.percentile(np.abs(all_diffs), 98)
print(f'Shared color scale: ±{shared_max:.3f}')

In [ ]:
# Plot all verticals as a grid: rows = vertical, cols = left/right lateral views
n_v = len(vertical_diffs)
fig, axes = plt.subplots(n_v, 2, figsize=(10, 3 * n_v),
                          subplot_kw={'projection': '3d'})
if n_v == 1:
    axes = axes.reshape(1, -1)

for row, (v, info) in enumerate(vertical_diffs.items()):
    diff = info['diff']
    left = diff[:n_per_hemi]
    right = diff[n_per_hemi:]
    plotting.plot_surf_stat_map(
        fsaverage.infl_left, left,
        hemi='left', view='lateral',
        bg_map=fsaverage.sulc_left,
        colorbar=False, cmap='RdBu_r',
        vmax=shared_max,
        threshold=shared_max * 0.3,
        axes=axes[row, 0], figure=fig,
    )
    plotting.plot_surf_stat_map(
        fsaverage.infl_right, right,
        hemi='right', view='lateral',
        bg_map=fsaverage.sulc_right,
        colorbar=(row == 0),
        cmap='RdBu_r',
        vmax=shared_max,
        threshold=shared_max * 0.3,
        axes=axes[row, 1], figure=fig,
    )
    axes[row, 0].set_title(f'{v} (n={info["n_viral"]}v/{info["n_dud"]}d) — left',
                            fontsize=11)
    axes[row, 1].set_title('right', fontsize=11)

plt.suptitle('Viral − Dud activation by vertical (shared scale, ±{:.2f})'.format(shared_max),
              fontsize=13, y=0.995)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'per_vertical_diff_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSaved {OUTPUT_DIR / "per_vertical_diff_maps.png"}')

## 4. Summary

In [ ]:
print('=' * 60)
print('PER-VERTICAL SUMMARY')
print('=' * 60)
print()
valid = per_vertical.dropna(subset=['delta']).copy()
if len(valid) > 0:
    best = valid.iloc[valid['delta'].argmax()]
    worst = valid.iloc[valid['delta'].argmin()]
    print(f'Largest TRIBE lift: {best["vertical"]} (delta = {best["delta"]:+.3f}, '
          f'baseline = {best["auc_baseline"]:.3f}, '
          f'baseline+TRIBE = {best["auc_tribe"]:.3f})')
    print(f'Smallest TRIBE lift: {worst["vertical"]} (delta = {worst["delta"]:+.3f}, '
          f'baseline = {worst["auc_baseline"]:.3f}, '
          f'baseline+TRIBE = {worst["auc_tribe"]:.3f})')
    n_positive = (valid['delta'] > 0).sum()
    print(f'\nTRIBE adds value in {n_positive}/{len(valid)} verticals.')
    print()
    print('Reminder for the writeup: n is small per vertical (~5v/5d).')
    print('Frame these as directional patterns, not validated AUCs.')
else:
    print('No verticals had enough samples for per-vertical AUC.')
print()
print(f'Diff maps computed for {len(vertical_diffs)} verticals.')
print(f'Output files: per_vertical_auc.csv, per_vertical_auc.png, per_vertical_diff_maps.png')